# Практическая работа 2. Работа с LLM через API
## Основное решение: расчет точности и сравнительный анализ техник промптирования

- **Студент:** Григорьев Михаил Романович
- **Группа:** ИСТ.04
- **Вариант:** 26 (`rental_26.csv`)
- **Дата:** 20.11.2026

### 1. Настройка окружения и подключение к GigaChat API

In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv

# Загрузка переменных окружения
load_dotenv()
GIGA_KEY = os.getenv('GIGA_KEY')

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Инициализация модели GigaChat
llm = GigaChat(
    credentials=GIGA_KEY,
    model='GigaChat-2',
    verify_ssl_certs=False,
    temperature=0.1,
    max_tokens=1000
)
print('Окружение успешно настроено. GigaChat инициализирован.')

Окружение успешно настроено. GigaChat инициализирован.


### 2. Загрузка данных варианта 26 в pandas DataFrame (слайд 18)

In [2]:
# Загрузка данных из csv в датафрейм df
df = pd.read_csv('rental_26.csv', sep=';')
print('Размер датасета:', df.shape)
print(df.head())

Размер датасета: (15, 3)
   amount                                                                                                                                                text result
0       2                                                Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийская деревня или рядом ,предложения в лс.      2
1       6                      Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный эконом. 3 взрослых и 3 детей (2,9,11 лет). Строго с 20 по 30 июля      6
2       2                                                                           Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17.07 по 27.07.      2
3       3                                                                  Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28.06 не далеко от моря.      3
4       4  Добрый День!! Семья 4 человека, 2 взрослых, дети 10 и 3 года, ищем жилье, можно 3-х местный с доп.местом. С 1 июля поближе к морю и не доро

### 3. Определение базового промпта и цепочки LangChain

In [3]:
# Базовый промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=['text'],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

### 4. Автоматическая обработка заявок в цикле и сохранение результатов (слайд 19)

In [4]:
results = []
for _, row in df.iterrows():
    text = row['text']
    try:
        time.sleep(1.2) # предотвращение RateLimitError (429)
        result = chain.invoke({'text': text})
        results.append(result.strip())
    except Exception as e:
        results.append(f'ERROR: {e}')

df['result'] = results
df.to_csv('rental_with_results.csv', index=False, encoding='utf-8-sig')
print('Обработка завершена. Результаты сохранены в rental_with_results.csv')
print(df[['amount', 'result', 'text']])

Обработка завершена. Результаты сохранены в rental_with_results.csv
    amount result                                                                                                                                                                                               text
0        2      2                                                                                               Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийская деревня или рядом ,предложения в лс.
1        6      6                                                                     Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный эконом. 3 взрослых и 3 детей (2,9,11 лет). Строго с 20 по 30 июля
2        2      2                                                                                                                          Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17.07 по 27.07.
3        3      3                                                               

### 5. Оценка качества и точности модели (слайды 20-21)

In [5]:
# Преобразование строки в числа с обработкой ошибок
numeric_result = pd.to_numeric(df['result'], errors='coerce')

total = len(df)
correct = (numeric_result == df['amount']).sum()
errors = total - correct
accuracy = correct / total

print(f'Ошибок: {errors}')
print(f'Верных ответов: {correct} из {total}')
print(f'Точность: {accuracy:.1%}')

Ошибок: 0
Верных ответов: 15 из 15
Точность: 100.0%


### 6. Сравнение различных техник промптирования (слайд 29)

In [6]:
# Определение продвинутых промптов
detailed_prompt = PromptTemplate(
    input_variables=['text'],
    template="""
Ты — эксперт по анализу текстов заявок на аренду жилья. Твоя задача — извлечь точное количество людей, которые будут проживать в арендуемом жилье.
ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки
2. Найди все упоминания количества людей (прямые и косвенные)
3. Учти следующие правила:
 - 'семья с одним ребенком' = 3 человека (родители + ребенок)
 - 'семья с двумя детьми' = 4 человека (родители + дети)
 - 'пара', 'двое', 'молодая семья' = 2 человека
 - 'один', 'студент', 'индивидуально' = 1 человек
4. Если количество указано числом (например, 'троих', 'четверых'), используй это число
5. Верни ТОЛЬКО целое число без дополнительных символов или пояснений
Текст заявки: {text}
Количество проживающих:"""
)

few_shot_prompt = PromptTemplate(
    input_variables=['text'],
    template="""
Извлеки количество людей из текста заявки на аренду жилья. Изучи примеры:
ПРИМЕРЫ:
Заявка: 'Семья из трех человек ищет двухкомнатную квартиру'
Количество: 3
Заявка: 'Молодая пара снимет студию в центре города'
Количество: 2
Заявка: 'Ищу однокомнатную квартиру для себя'
Количество: 1
Заявка: 'Семья с одним ребенком рассматривает варианты жилья'
Количество: 3
Заявка: 'Четверо студентов ищут четырехкомнатную квартиру'
Количество: 4
Теперь проанализируй эту заявку:
Заявка: {text}
Количество:"""
)

test_cases = [
    'Семья из пяти человек ищет большую квартиру',
    'Молодая пара с ребенком рассматривает двухкомнатную квартиру',
    'Сниму жилье эконом для одного студента на неделю',
    'Едем отдыхать: двое взрослых и трое детей'
]

techniques = {
    'Базовый': basic_prompt | llm | StrOutputParser(),
    'Детальный': detailed_prompt | llm | StrOutputParser(),
    'Few-shot': few_shot_prompt | llm | StrOutputParser()
}

for text in test_cases:
    print(f'\nТекст: {text}')
    for name, chain_t in techniques.items():
        time.sleep(1.2)
        try:
            res = chain_t.invoke({'text': text})
            print(f'{name}: {res.strip()}')
        except Exception as e:
            print(f'{name}: Ошибка - {e}')


Текст: Семья из пяти человек ищет большую квартиру
Базовый: 5
Детальный: 5
Few-shot: Количество: 5

Текст: Молодая пара с ребенком рассматривает двухкомнатную квартиру
Базовый: 3
Детальный: 3
Few-shot: Количество: 3

Текст: Сниму жилье эконом для одного студента на неделю
Базовый: 1
Детальный: 1
Few-shot: Количество: 1

Текст: Едем отдыхать: двое взрослых и трое детей
Базовый: 5
Детальный: 5
Few-shot: Количество: 5


---
# Задание № 2: Продвинутые техники и структурированный JSON-вывод

В данном разделе решается задача комплексного извлечения атрибутов заявки:
1. `count_adults` (число взрослых) и `count_children` (число детей)
2. `start_date` — дата заезда (или самая ранняя дата при плавающих диапазонах)
3. `nights` — число ночей проживания (или максимальное число дней при диапазонах)
4. `price_per_day` — желаемая цена в сутки (при диапазоне берется максимум)
5. `remarks` — особые пожелания (наличие моря, бассейна, животных, санузла и др.)

Используется связка **ChatPromptTemplate (System + Human)**, **Few-Shot промптирование** и **JsonOutputParser** на базе схемы Pydantic.

In [7]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.output_parsers import JsonOutputParser

# Описание схемы извлекаемых данных с валидацией через Pydantic
class RentalDetails(BaseModel):
    count_adults: Optional[int] = Field(description='Число взрослых (целое число или null)')
    count_children: Optional[int] = Field(description='Число детей (целое число или null)')
    start_date: Optional[str] = Field(description='Дата заезда в формате DD.MM или DD.MM.YYYY (или самая ранняя дата при плавающих диапазонах, либо null)')
    nights: Optional[int] = Field(description='Число ночей проживания (или максимальное число дней при диапазонах, либо null)')
    price_per_day: Optional[int] = Field(description='Желаемая цена в сутки в рублях (при диапазоне брать максимум, либо null)')
    remarks: Optional[str] = Field(description='Особые пожелания (море, бассейн, кондиционер, животные и др.)')

parser = JsonOutputParser(pydantic_object=RentalDetails)

# Системное сообщение с правилами и примерами (Few-shot)
system_text = """Ты — эксперт по извлечению структурированных данных из объявлений и заявок на аренду жилья.
Твоя задача — извлечь точные параметры бронирования из заявки:
1. count_adults (число взрослых)
2. count_children (число детей)
3. start_date (дата заезда, формат DD.MM или DD.MM.YYYY, при плавающих диапазонах — самая ранняя дата)
4. nights (число ночей проживания; если дан интервал дат 'с Д1 по Д2', число ночей = дата_выезда - дата_заезда; при интервале дней '7-10 дней' возьми максимум, т.е. 10; если срок не указан — null)
5. price_per_day (желаемая цена в сутки в рублях, при диапазоне 'до 1000' или '1000-1500' бери максимум, если цена не указана — null)
6. remarks (особые пожелания: море, бассейн, кондиционер, животные, санузел в номере и др.)

ПРИМЕРЫ (Few-Shot):
Пример 1:
Заявка: 'Семья из трех человек (2 взрослых и ребенок 5 лет) ищет квартиру с 10 по 20 июля, до 3000 руб в сутки, нужен кондиционер.'
Ответ:
{{"count_adults": 2, "count_children": 1, "start_date": "10.07", "nights": 10, "price_per_day": 3000, "remarks": "нужен кондиционер"}}

Пример 2:
Заявка: 'Сниму 2-местный номер на двоих на 7-10 дней с 15 августа. Близко к морю.'
Ответ:
{{"count_adults": 2, "count_children": 0, "start_date": "15.08", "nights": 10, "price_per_day": null, "remarks": "близко к морю, 2-местный номер"}}

Отвечай строго в формате JSON без дополнительного текста.
{format_instructions}"""

human_text = "Заявка: {text}"

chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_text),
    HumanMessagePromptTemplate.from_template(human_text)
]).partial(format_instructions=parser.get_format_instructions())

structured_chain = chat_prompt | llm | parser
print('Схема Pydantic и ChatPromptTemplate успешно определены.')

Схема Pydantic и ChatPromptTemplate успешно определены.


### 8. Извлечение структурированных полей из 15 заявок варианта 26

In [8]:
predictions = []
for idx, row in df.iterrows():
    time.sleep(1.2) # выдерживаем паузу для соблюдения Rate Limit
    res = structured_chain.invoke({'text': row['text']})
    predictions.append(res)

df_structured = pd.DataFrame(predictions)
df_combined = pd.concat([df, df_structured], axis=1)
df_combined.to_csv('rental_structured_results.csv', index=False, encoding='utf-8-sig')
print(f'Успешно обработано {len(predictions)} заявок:')
print(df_combined[['count_adults', 'count_children', 'start_date', 'nights', 'price_per_day', 'remarks']])

Успешно обработано 15 заявок:
    count_adults  count_children  start_date  nights  price_per_day                                                                                                            remarks
0            2.0             0.0       01.09     7.0            NaN                                                                         су в номере, олимпийская деревня или рядом
1            3.0             3.0       20.07    11.0            NaN                          недорогое жилье, недалеко от моря, 3-местный и 2-местный эконом, возраст детей 2,9,11 лет
2            2.0             0.0       17.07    11.0            NaN                                                                                     2х местный номер, с удобствами
3            NaN             NaN       20.06     8.0            NaN                                                                                  3 местный номер, недалеко от моря
4            2.0             2.0       01.07     NaN   

### 9. Сравнение с ручной разметкой и расчет точности по каждому полю (1-4)

In [9]:
# Эталонная ручная разметка для 15 заявок варианта 26
ground_truth = [
    {'count_adults': 2, 'count_children': 0, 'start_date': '01.09', 'nights': 7, 'price_per_day': None},
    {'count_adults': 3, 'count_children': 3, 'start_date': '20.07', 'nights': 10, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 0, 'start_date': '17.07', 'nights': 10, 'price_per_day': None},
    {'count_adults': 3, 'count_children': 0, 'start_date': '20.06', 'nights': 8, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 2, 'start_date': '01.07', 'nights': None, 'price_per_day': None},
    {'count_adults': 3, 'count_children': 0, 'start_date': '04.08', 'nights': 10, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 0, 'start_date': '07.08', 'nights': 8, 'price_per_day': 1000},
    {'count_adults': 2, 'count_children': 1, 'start_date': '10.08.2018', 'nights': 16, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 1, 'start_date': '15.08', 'nights': 8, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 3, 'start_date': '20.09', 'nights': 10, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 0, 'start_date': '24.07', 'nights': 8, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 2, 'start_date': '08.08', 'nights': 13, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 2, 'start_date': '08.07', 'nights': 6, 'price_per_day': None},
    {'count_adults': 1, 'count_children': 0, 'start_date': '10.07', 'nights': 10, 'price_per_day': None},
    {'count_adults': 2, 'count_children': 2, 'start_date': '29.06.19', 'nights': None, 'price_per_day': None}
]

def clean_date(d):
    if not d or pd.isna(d): return ''
    return str(d).replace('.2018', '').replace('.2019', '').replace('.19', '').strip()

correct_people = 0
correct_adults = 0
correct_children = 0
correct_date = 0
correct_nights = 0
correct_price = 0
total = len(df)

for i in range(total):
    gt = ground_truth[i]
    pr = predictions[i]
    a_ok = (pr.get('count_adults') == gt['count_adults'])
    c_ok = (pr.get('count_children') == gt['count_children'])
    if a_ok: correct_adults += 1
    if c_ok: correct_children += 1
    if a_ok and c_ok: correct_people += 1
    if clean_date(pr.get('start_date')) == clean_date(gt['start_date']):
        correct_date += 1
    if pr.get('nights') == gt['nights']:
        correct_nights += 1
    if pr.get('price_per_day') == gt['price_per_day']:
        correct_price += 1

acc_people = correct_people / total
acc_adults = correct_adults / total
acc_children = correct_children / total
acc_date = correct_date / total
acc_nights = correct_nights / total
acc_price = correct_price / total
avg_acc = (acc_people + acc_date + acc_nights + acc_price) / 4.0

print('МЕТРИКИ ТОЧНОСТИ ИЗВЛЕЧЕНИЯ ПО ПОЛЯМ (Задание 2):')
print(f'1. Взрослые и дети (count_adults & count_children): {correct_people}/{total} ({acc_people:.1%})')
print(f'   - count_adults: {correct_adults}/{total} ({acc_adults:.1%})')
print(f'   - count_children: {correct_children}/{total} ({acc_children:.1%})')
print(f'2. Дата заезда (start_date): {correct_date}/{total} ({acc_date:.1%})')
print(f'3. Число ночей (nights): {correct_nights}/{total} ({acc_nights:.1%})')
print(f'4. Цена в сутки (price_per_day): {correct_price}/{total} ({acc_price:.1%})')
print('-' * 60)
print(f'Средняя точность по всем показателям (1-4): {avg_acc:.1%}')

МЕТРИКИ ТОЧНОСТИ ИЗВЛЕЧЕНИЯ ПО ПОЛЯМ (Задание 2):
1. Взрослые и дети (count_adults & count_children): 14/15 (93.3%)
   - count_adults: 14/15 (93.3%)
   - count_children: 14/15 (93.3%)
2. Дата заезда (start_date): 15/15 (100.0%)
3. Число ночей (nights): 9/15 (60.0%)
4. Цена в сутки (price_per_day): 15/15 (100.0%)
------------------------------------------------------------
Средняя точность по всем показателям (1-4): 88.3%
